# Eco-Logistics GRPO Training — v2 (hackathon submission)

**Theme:** #3.1 World Modeling / Professional Tasks (primary), #2 Long-Horizon Planning (secondary).

**What this notebook does:**
1. Installs TRL + OpenEnv + Unsloth
2. Connects to the hosted `eco-logistics` HF Space (configured for parallel rollouts)
3. Wraps the env with **non-stationary demand shocks** and a **competitor agent** that bids up shipping prices — strengthens the world-modeling story without pretending to be multi-agent
4. Normalizes the full dense reward (profit − shipping − carbon − storage + healthy-stock bonus) for GRPO
5. Logs 4 metrics per episode: total reward, profit, carbon usage, delivery success rate
6. Runs GRPO training
7. Produces 2 training curves (reward vs episodes, carbon efficiency vs episodes) for the pitch deck
8. Compares before/after agent behavior

**Hardware:** single T4 (free Colab) or A100 (HF credits). Uses Unsloth + LoRA to fit in <15GB VRAM.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q "transformers>=4.56.0,<5.0" "trl[vllm]==0.24.0" openenv-core unsloth "pandas<3.0.0" "numpy<2.4" matplotlib "huggingface-hub>=0.34.0,<1.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.4/436.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/

In [3]:
from huggingface_hub import login
import os

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    print("WARNING: set HF_TOKEN before pushing the final model.")

## 2. Configuration

In [4]:
# ---- Point this at your deployed HF Space ----
# IMPORTANT: your main.py on the Space MUST set max_concurrent_envs >= REQUIRED_CONCURRENT
# (printed below). With the current config that's 16, not 8.
ENV_URL = os.environ.get("ENV_URL", "https://lokeshrao226-eco-logistics.hf.space")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Primary training task. Also re-run on "net_zero_profit" for the hard-task story.
TASK_ID = "inventory_balanced"

# Tasks we sample across for diverse training prompts.
# Set to [TASK_ID] if your env doesn't expose all three.
TRAIN_TASK_IDS = ["restock_only", "inventory_balanced", "net_zero_profit"]

# GRPO hyperparameters
NUM_GENERATIONS = 4
PER_DEVICE_BATCH = 1
GRAD_ACCUM_STEPS = 4
MAX_STEPS = 100
# NOTE: max_completion_length is set in GRPOConfig (cell 23) to 2048 — a
# 10-object JSON array needs ~500-700 tokens. Don't redefine it here.
MAX_ENV_STEPS = 10
LEARNING_RATE = 5e-6

# Dataset diversity — NEW in Tier 1 patch
N_UNIQUE_PROMPTS = 50       # distinct initial-state prompts (was: 10)
TRAIN_SEED_RANGE = (0, 50)  # inclusive start, exclusive end
EVAL_SEED_RANGE  = (500, 510)  # held out for BEFORE/AFTER eval

# World-modeling wrapper knobs (client-side, no env changes needed)
DEMAND_SHOCK_PROB = 0.15
DEMAND_SHOCK_MULT = 2.5
COMPETITOR_BID_PROB = 0.20
COMPETITOR_BID_MULT = 1.8

# Concurrency guard
REQUIRED_CONCURRENT = NUM_GENERATIONS * PER_DEVICE_BATCH * GRAD_ACCUM_STEPS
print(f"HF Space must support max_concurrent_envs >= {REQUIRED_CONCURRENT}")
print(f"Training {MODEL_NAME} on task={TASK_ID} against {ENV_URL}")
print(f"Dataset: {N_UNIQUE_PROMPTS} unique prompts across {len(TRAIN_TASK_IDS)} task(s)")


HF Space must support max_concurrent_envs >= 16
Training Qwen/Qwen2.5-1.5B-Instruct on task=inventory_balanced against https://lokeshrao226-eco-logistics.hf.space


## 3. Smoke-test the env

In [5]:
import requests
import time as _time
import uuid # Added for session_id generation

def _post_with_retry(url, *, json=None, headers=None, timeout=30,
                     max_retries=6, base_delay=2.0):
    delay = base_delay
    for attempt in range(max_retries + 1):
        r = requests.post(url, json=json, headers=headers, timeout=timeout)

        if r.status_code not in (429, 500, 502, 503):
            r.raise_for_status()
            return r

        retry_after = r.headers.get("Retry-After")
        wait = float(retry_after) if retry_after else min(delay, 60.0)
        if attempt < max_retries:
            print(f"  [retry {r.status_code}] {url.rsplit('/',1)[-1]} — "
                  f"waiting {wait:.1f}s (attempt {attempt+1}/{max_retries})")
            _time.sleep(wait)
            delay *= 2
        else:
            r.raise_for_status()
    return r # unreachable, but keeps linters happy

# 1. Initialize headers for the session
session_id = f"test-{uuid.uuid4().hex[:12]}"
headers = {"X-Session-Id": session_id}

# 2. Reset the environment first (logical order for a smoke test)
r = _post_with_retry(f"{ENV_URL}/reset", json={"task_id": TASK_ID},
                     headers=headers, timeout=30)
initial_obs = r.json()
print("Reset OK. Observation keys:", list(initial_obs.keys()))

# 3. Define a dummy action
dummy_action = {
    "ship_amount": 0.0,
    "origin_city": "Seattle",
    "destination_city": "Chicago",
    "speed_mode": "Rail",
}

# 4. Take a step with the dummy action
r = _post_with_retry(f"{ENV_URL}/step", json=dummy_action,
                     headers=headers, timeout=30)
step_resp = r.json()
print("Step OK. Reward breakdown:", step_resp.get("reward"))

Reset OK. Observation keys: ['current_inventory', 'pending_shipments', 'current_demand', 'carbon_credit_balance', 'step_number', 'total_steps', 'weather_alert', 'cumulative_profit', 'cumulative_carbon']
Step OK. Reward breakdown: {'sales_revenue': 431.0, 'shipping_cost': 0.0, 'carbon_penalty': 0.0, 'storage_fee': 96.65, 'healthy_stock_bonus': 0.1, 'total': 334.45}


## 4. World-modeling wrapper: non-stationary demand + competitor

These two knobs add genuine world-modeling complexity on top of the base env. They run **client-side** so we don't need to redeploy the Space:

- **Demand shock**: with probability `DEMAND_SHOCK_PROB` per step, multiply the observed `current_demand` by `DEMAND_SHOCK_MULT`. The agent sees the spike in its prompt and must react (ship more, use Air mode, etc).
- **Competitor bid**: with probability `COMPETITOR_BID_PROB`, a random route becomes 1.8x more expensive for that step. We surface this as a `weather_alert`-style warning so the agent has to route around it.

These change the **observation the agent sees** and the **reward it receives** after the env responds — they do NOT require modifying the environment itself. This keeps us Round-1-compliant while honestly increasing task difficulty.

In [6]:
import random

CITIES = ["Seattle", "Chicago", "NYC"]

def apply_world_wrapper(obs: dict, rng: random.Random) -> tuple[dict, dict]:
    """Apply demand shock + competitor bid to an observation before showing it to the agent.

    Returns (modified_obs, shock_state) where shock_state holds the per-step modifiers
    we'll use to adjust the reward post-hoc.
    """
    modified = dict(obs)
    shock_state = {"demand_shocked": False, "contested_route": None}

    # Demand shock
    if rng.random() < DEMAND_SHOCK_PROB:
        shocked_demand = {c: v * DEMAND_SHOCK_MULT for c, v in obs["current_demand"].items()}
        modified["current_demand"] = shocked_demand
        shock_state["demand_shocked"] = True

    # Competitor bid — surface as a weather_alert-style note
    if rng.random() < COMPETITOR_BID_PROB:
        origin = rng.choice(CITIES)
        dest = rng.choice([c for c in CITIES if c != origin])
        shock_state["contested_route"] = (origin, dest)
        existing = modified.get("weather_alert") or ""
        note = f"Competitor bid up {origin}→{dest} route ({COMPETITOR_BID_MULT}x cost this step)"
        modified["weather_alert"] = f"{existing}; {note}" if existing else note

    return modified, shock_state


def apply_reward_wrapper(reward_dict: dict, action: dict, shock_state: dict) -> dict:
    """Adjust reward components based on the shocks that were active this step.
    Returns a modified reward dict with a new 'total' field.
    """
    adjusted = dict(reward_dict)

    # If the agent shipped on a contested route, penalize extra shipping cost
    contested = shock_state.get("contested_route")
    if contested and action.get("ship_amount", 0) > 0:
        if (action.get("origin_city"), action.get("destination_city")) == contested:
            extra_cost = adjusted.get("shipping_cost", 0) * (COMPETITOR_BID_MULT - 1.0)
            adjusted["shipping_cost"] = adjusted.get("shipping_cost", 0) + extra_cost

    # Recompute total with the same formula the env uses
    adjusted["total"] = (
        adjusted.get("sales_revenue", 0)
        - adjusted.get("shipping_cost", 0)
        - adjusted.get("carbon_penalty", 0)
        - adjusted.get("storage_fee", 0)
        + adjusted.get("healthy_stock_bonus", 0)
    )
    return adjusted

## 5. System prompt + action parser

In [7]:
SYSTEM_PROMPT = """You are an AI supply chain manager for a 3-warehouse network \
(Seattle, Chicago, NYC).

You are planning an ENTIRE {n}-step episode UPFRONT. The environment will inject \
non-stationary demand shocks and competitor bids mid-episode; your plan must be \
robust to these surprises before they arrive.

Output EXACTLY one JSON array containing EXACTLY {n} action objects — one per step \
in order:
[
  {{"ship_amount": <float ≥ 0>, "origin_city": <"Seattle"|"Chicago"|"NYC">,
    "destination_city": <"Seattle"|"Chicago"|"NYC">, "speed_mode": <"Air"|"Rail">}},
  ... (repeat for all {n} steps)
]

Planning rules:
- Rail: cheap, low-carbon, slow (3-step transit). Prefer for routine restocking.
- Air: fast (1-step), expensive, high-carbon. Reserve for demand-spike response.
- ship_amount=0 is a valid no-op — use it to hold buffer stock against shocks.
- Defensive routing: spread shipments across multiple origin→destination pairs so
  a single contested route cannot block all supply.
- If weather_alert warns of a disruption, plan an alternative route in later steps.

Respond with ONLY the JSON array. No explanation, no markdown, no extra text.""".format(
    n=MAX_ENV_STEPS
)


def format_observation_prompt(obs: dict) -> str:
    return (
        f"Initial state — plan all {obs['total_steps']} steps now.\n"
        f"Step {obs['step_number']}/{obs['total_steps']}\n"
        f"Inventory: {obs['current_inventory']}\n"
        f"Demand forecast (step 0): {obs['current_demand']}\n"
        f"Pending shipments: {obs.get('pending_shipments', [])}\n"
        f"Carbon credits left: {obs['carbon_credit_balance']:.1f}\n"
        f"Cumulative profit: {obs.get('cumulative_profit', 0):.2f}\n"
        f"Weather/market alert: {obs.get('weather_alert') or 'clear'}\n"
        f"\nYour 10-step plan (JSON array only):"
    )

In [8]:
# ── Cell 46 ──────────────────────────────────────────────────────────────────
import json
import re

# Bug fix: use dict(...) inside the comprehension so each slot is a distinct
# dict. Previously all 10 slots pointed to the same dict reference.
SAFE_FALLBACK_ACTIONS: list[dict] = [
    {
        "ship_amount": 0.0,
        "origin_city": "Seattle",
        "destination_city": "Chicago",
        "speed_mode": "Rail",
    }
    for _ in range(MAX_ENV_STEPS)
]
# Force independence (defensive — in case callers mutate entries)
SAFE_FALLBACK_ACTIONS = [dict(a) for a in SAFE_FALLBACK_ACTIONS]

_REQUIRED_KEYS = {"ship_amount", "origin_city", "destination_city", "speed_mode"}
_VALID_CITIES  = {"Seattle", "Chicago", "NYC"}
_VALID_SPEEDS  = {"Air", "Rail"}


def parse_action_array(completion: str) -> tuple[list[dict], bool]:
    m = re.search(r"\[.*\]", completion, re.DOTALL)
    if not m:
        return [dict(a) for a in SAFE_FALLBACK_ACTIONS], False
    try:
        arr = json.loads(m.group(0))
    except (json.JSONDecodeError, ValueError):
        return [dict(a) for a in SAFE_FALLBACK_ACTIONS], False

    if not isinstance(arr, list) or len(arr) != MAX_ENV_STEPS:
        return [dict(a) for a in SAFE_FALLBACK_ACTIONS], False

    for item in arr:
        if not isinstance(item, dict):
            return [dict(a) for a in SAFE_FALLBACK_ACTIONS], False
        if not _REQUIRED_KEYS.issubset(item.keys()):
            return [dict(a) for a in SAFE_FALLBACK_ACTIONS], False
        if item["origin_city"] not in _VALID_CITIES:
            return [dict(a) for a in SAFE_FALLBACK_ACTIONS], False
        if item["destination_city"] not in _VALID_CITIES:
            return [dict(a) for a in SAFE_FALLBACK_ACTIONS], False
        if item["origin_city"] == item["destination_city"]:
            return [dict(a) for a in SAFE_FALLBACK_ACTIONS], False
        if item["speed_mode"] not in _VALID_SPEEDS:
            return [dict(a) for a in SAFE_FALLBACK_ACTIONS], False
        if not isinstance(item["ship_amount"], (int, float)) or item["ship_amount"] < 0:
            return [dict(a) for a in SAFE_FALLBACK_ACTIONS], False

    return arr, True


## 6. Episode rollout with full metric logging

Returns the 4 metrics the judges want to see:
- `total_reward` — summed normalized reward (what GRPO optimizes)
- `profit` — summed sales_revenue − shipping_cost (business metric)
- `carbon_used` — cumulative carbon emissions (sustainability metric)
- `delivery_success_rate` — fraction of demand fulfilled (operations metric)
- `valid_action_rate` — fraction of model outputs that parsed as valid JSON (diagnostic)

In [9]:
# ── Replace the entire run_episode function (Section 6 cell) ─────────────────

def run_episode(
    generate_fn,
    task_id: str = TASK_ID,
    max_steps: int = MAX_ENV_STEPS,
    env_url: str = ENV_URL,
    seed: int | None = None,
):
    """Run one full episode under the Upfront Trajectory Planning protocol.

    1. Reset env → get initial observation.
    2. Call generate_fn ONCE on that observation → get a 10-action JSON array.
    3. Execute parsed_actions[step_idx] at every step (no heuristic fill-in).
    4. apply_world_wrapper / apply_reward_wrapper still fire each step.

    Returns the same metrics dict as before so summarize() and the plots are
    unaffected.
    """
    rng        = random.Random(seed)
    session_id = f"train-{uuid.uuid4().hex[:12]}"
    headers    = {"X-Session-Id": session_id}

    r = _post_with_retry(
        f"{env_url}/reset", json={"task_id": task_id},
        headers=headers, timeout=30
    )
    obs = r.json()

    # ── Generate the full 10-step plan from the initial observation ────────────
    wrapped_obs_0, _ = apply_world_wrapper(obs, rng)
    user_prompt = format_observation_prompt(wrapped_obs_0)
    full_prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}\n"
        f"<|user|>\n{user_prompt}\n"
        f"<|assistant|>\n"
    )
    completion = generate_fn(full_prompt)
    parsed_actions, was_valid = parse_action_array(completion)  # ← fixed call

    # ── Execute the plan ───────────────────────────────────────────────────────
    trajectory     = []
    total_reward   = 0.0
    total_profit   = 0.0
    total_carbon   = 0.0
    total_demand   = 0.0
    total_fulfilled = 0.0
    UNIT_PRICE     = 10.0

    for step_idx in range(max_steps):
        wrapped_obs, shock_state = apply_world_wrapper(obs, rng)

        # No heuristic — the model's own plan runs every step
        action = parsed_actions[step_idx]

        r = _post_with_retry(
            f"{env_url}/step", json=action,
            headers=headers, timeout=30
        )
        step_resp       = r.json()
        raw_reward      = step_resp.get("reward", {})
        adjusted_reward = apply_reward_wrapper(raw_reward, action, shock_state)
        step_reward     = adjusted_reward.get("total", 0.0)

        total_reward   += step_reward
        total_profit   += (adjusted_reward.get("sales_revenue", 0)
                           - adjusted_reward.get("shipping_cost", 0))
        total_carbon   += adjusted_reward.get("carbon_penalty", 0)

        step_demand      = sum(wrapped_obs["current_demand"].values())
        total_demand    += step_demand
        total_fulfilled += adjusted_reward.get("sales_revenue", 0) / UNIT_PRICE

        obs = step_resp.get("observation", obs)
        trajectory.append({
            "step":      step_idx,
            "action":    action,
            "valid":     was_valid,        # plan-level validity, same for every step
            "reward":    step_reward,
            "profit":    adjusted_reward.get("sales_revenue", 0) - adjusted_reward.get("shipping_cost", 0),
            "carbon":    adjusted_reward.get("carbon_penalty", 0),
            "shocked":   shock_state["demand_shocked"],
            "contested": shock_state["contested_route"] is not None,
        })

        if step_resp.get("done"):
            break

    # Grader score
    try:
        g = _post_with_retry(
            f"{env_url}/grader", json={"task_id": task_id},
            headers=headers, timeout=30
        )
        grader_score = g.json().get("score", 0.0)
    except Exception:
        grader_score = 0.0

    return {
        "total_reward":          total_reward,
        "profit":                total_profit,
        "carbon_used":           total_carbon,
        "carbon_efficiency":     total_profit / max(1.0, total_carbon),
        "delivery_success_rate": total_fulfilled / max(1.0, total_demand),
        "valid_action_rate":     float(was_valid),   # 1.0 or 0.0 per episode
        "grader_score":          grader_score,
        "trajectory":            trajectory,
    }

## 7. Load model + baseline eval

In [10]:
# torchcodec conflicts with unsloth's torch build — remove it before loading.
# If this is the first time in the session, we need to restart the runtime
# after uninstalling so the import cache is cleared.
import importlib, sys, subprocess

_had_torchcodec = "torchcodec" in sys.modules
subprocess.run(["pip", "uninstall", "-y", "torchcodec"], check=False)

if _had_torchcodec:
    print("torchcodec was loaded. RESTART THE RUNTIME now, then re-run from cell 18.")
else:
    print("torchcodec uninstalled (not previously imported — safe to continue).")


Found existing installation: torchcodec 0.10.0+cu128
Uninstalling torchcodec-0.10.0+cu128:
  Successfully uninstalled torchcodec-0.10.0+cu128


In [11]:
from unsloth import FastLanguageModel
import torch
import numpy as np

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=3072,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 04-23 15:55:52 [__init__.py:216] Automatically detected platform cuda.
ERROR 04-23 15:55:53 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.10.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [12]:
def eager_generate(prompt: str) -> str:
    """Inference-mode generate. SAFE ONLY OUTSIDE GRPO TRAINING.

    Calling FastLanguageModel.for_inference during training corrupts LoRA
    adapter state. Use this for baseline eval and post-training eval only.
    """
    FastLanguageModel.for_inference(model)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=1024,   # 10-action JSON array needs ~500-700 tokens
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    completion = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return completion


# ─── Additional baselines: random policy + heuristic policy ─────────────────
# These let us show that the improvement isn't just "base Qwen sucks at JSON".

def random_policy_generate(prompt: str, rng=None) -> str:
    """Emits a valid 10-action JSON array with random choices."""
    import random as _r
    rng = rng or _r.Random()
    actions = []
    for _ in range(MAX_ENV_STEPS):
        o = rng.choice(list(_VALID_CITIES))
        d = rng.choice([c for c in _VALID_CITIES if c != o])
        actions.append({
            "ship_amount": round(rng.uniform(0, 30), 1),
            "origin_city": o,
            "destination_city": d,
            "speed_mode": rng.choice(list(_VALID_SPEEDS)),
        })
    return json.dumps(actions)


def heuristic_policy_generate(prompt: str) -> str:
    """Hand-coded rule-of-thumb: ship rail from highest-inv to lowest-inv city.

    Parses the inventory dict out of the prompt text. This is a reasonable
    non-learned baseline — matches what a careful engineer would write in 20 min.
    """
    # Extract inventory line from the prompt
    m = re.search(r"Inventory:\s*(\{[^}]+\})", prompt)
    actions = []
    if m:
        try:
            inv = json.loads(m.group(1).replace("'", '"'))
            # Order cities by inventory ascending; ship from richest to poorest
            cities_sorted = sorted(inv.items(), key=lambda kv: kv[1])
            poorest = cities_sorted[0][0]
            richest = cities_sorted[-1][0]
            amount = max(0.0, (inv[richest] - inv[poorest]) / 2)
            for _ in range(MAX_ENV_STEPS):
                actions.append({
                    "ship_amount": round(amount, 1),
                    "origin_city": richest,
                    "destination_city": poorest,
                    "speed_mode": "Rail",
                })
            return json.dumps(actions)
        except Exception:
            pass
    # Fallback
    return json.dumps([dict(a) for a in SAFE_FALLBACK_ACTIONS])


print("Running 10 BASELINE episodes (base Qwen, no training)...")
baseline_results = [run_episode(eager_generate, seed=500+i) for i in range(10)]

print("\nRunning 10 RANDOM-POLICY baseline episodes...")
import random as _rpy
_rng = _rpy.Random(123)
random_results = [run_episode(lambda p: random_policy_generate(p, _rng), seed=500+i) for i in range(10)]

print("\nRunning 10 HEURISTIC-POLICY baseline episodes...")
heuristic_results = [run_episode(heuristic_policy_generate, seed=500+i) for i in range(10)]


def summarize(results, label):
    print(f"\n=== {label} ===")
    for key in ["total_reward", "profit", "carbon_used", "carbon_efficiency",
                "delivery_success_rate", "valid_action_rate", "grader_score"]:
        vals = [r[key] for r in results]
        print(f"  {key:25s}: {np.mean(vals):+8.3f} ± {np.std(vals):.3f}")

summarize(random_results, "RANDOM POLICY")
summarize(heuristic_results, "HEURISTIC POLICY")
summarize(baseline_results, "BEFORE TRAINING (base Qwen)")


Running 10 BASELINE episodes...

=== BEFORE TRAINING ===
  total_reward             : +3697.120 ± 0.000
  profit                   : +4820.000 ± 0.000
  carbon_used              :   +0.000 ± 0.000
  carbon_efficiency        : +4820.000 ± 0.000
  delivery_success_rate    :   +0.757 ± 0.115
  valid_action_rate        :   +0.000 ± 0.000
  grader_score             :   +0.179 ± 0.000


## 8. GRPO training with normalized dense reward

Key points:
- **Reward normalization**: we divide episode returns by a running std to give GRPO stable advantages. GRPO's group-relative advantage already normalizes within a group; this extra normalization helps across groups.
- **Full dense signal preserved**: `total_reward` is computed from all four components (revenue − shipping − carbon − storage + bonus) plus our wrapper adjustments.
- **Per-step metric logging**: we record profit/carbon/delivery alongside reward so the training curves show tradeoffs, not just one number.

In [13]:
# ── Cell 56 — logging state + reward function ────────────────────────────────

if "global_step_counter" not in dir():
    global_step_counter = {"n": 0}

if "training_log" not in dir():
    training_log = {
        "grpo_step":        [],
        "mean_reward":      [],
        "mean_grader":      [],
        "mean_profit":      [],
        "mean_carbon":      [],
        "carbon_efficiency":[],
        "delivery_rate":    [],
        "valid_rate":       [],
    }

FALLBACK_ACTION = {
    "ship_amount": 0.0,
    "origin_city": "Seattle",
    "destination_city": "Chicago",
    "speed_mode": "Rail",
}


def env_reward_func(prompts, completions, **kwargs):
    rewards        = []
    batch_valid    = []
    batch_grader   = []
    batch_delivery = []
    batch_profit   = []
    batch_carbon   = []

    for completion in completions:

        parsed_actions, was_valid = parse_action_array(completion)

        session_id = f"grpo-{uuid.uuid4().hex[:12]}"
        headers    = {"X-Session-Id": session_id}

        total_reward         = 0.0
        total_profit_episode = 0.0
        total_carbon_episode = 0.0
        total_demand         = 0.0
        total_fulfilled      = 0.0
        grader_score         = 0.0
        UNIT_PRICE           = 10.0

        try:
            r = _post_with_retry(
                f"{ENV_URL}/reset", json={"task_id": TASK_ID},
                headers=headers, timeout=30
            )
            obs = r.json()

            for step_idx in range(MAX_ENV_STEPS):
                wrapped_obs, shock_state = apply_world_wrapper(
                    obs, random.Random(session_id + str(step_idx))
                )

                action = parsed_actions[step_idx]

                r = _post_with_retry(
                    f"{ENV_URL}/step", json=action,
                    headers=headers, timeout=30
                )
                raw_step_resp   = r.json()
                raw_reward_dict = raw_step_resp.get("reward", {})
                adjusted_reward = apply_reward_wrapper(raw_reward_dict, action, shock_state)

                total_reward         += adjusted_reward.get("total", 0.0)
                total_profit_episode += (
                    adjusted_reward.get("sales_revenue", 0)
                    - adjusted_reward.get("shipping_cost", 0)
                )
                total_carbon_episode += adjusted_reward.get("carbon_penalty", 0)

                demand           = sum(wrapped_obs.get("current_demand", {}).values())
                total_demand    += demand
                total_fulfilled += adjusted_reward.get("sales_revenue", 0.0) / UNIT_PRICE

                obs = raw_step_resp.get("observation", obs)
                if raw_step_resp.get("done"):
                    break

            try:
                g = _post_with_retry(
                    f"{ENV_URL}/grader", json={"task_id": TASK_ID},
                    headers=headers, timeout=30
                )
                grader_score = g.json().get("score", 0.0)
            except Exception:
                grader_score = 0.0

        except Exception as e:
            print(f"[env_reward_func] episode error: {e}")
            total_reward         = 0.0
            total_profit_episode = 0.0
            total_carbon_episode = 0.0
            grader_score         = 0.0

        format_bonus = 5.0 if was_valid else -5.0
        final_reward = total_reward + format_bonus

        rewards.append(float(final_reward))
        batch_valid.append(float(was_valid))
        batch_grader.append(grader_score)
        batch_delivery.append(total_fulfilled / max(1.0, total_demand))
        batch_profit.append(total_profit_episode)
        batch_carbon.append(total_carbon_episode)

    # ── Logging ───────────────────────────────────────────────────────────────
    global_step_counter["n"] += 1
    training_log["grpo_step"].append(global_step_counter["n"])
    training_log["mean_reward"].append(float(sum(rewards) / len(rewards)))
    training_log["mean_grader"].append(float(sum(batch_grader) / len(batch_grader)))
    training_log["delivery_rate"].append(float(sum(batch_delivery) / len(batch_delivery)))
    training_log["valid_rate"].append(float(sum(batch_valid) / len(batch_valid)))

    mean_profit = sum(batch_profit) / len(batch_profit) if batch_profit else 0.0
    mean_carbon = sum(batch_carbon) / len(batch_carbon) if batch_carbon else 0.0
    training_log["mean_profit"].append(mean_profit)
    training_log["mean_carbon"].append(mean_carbon)
    training_log["carbon_efficiency"].append(mean_profit / max(1.0, mean_carbon))

    if global_step_counter["n"] % 5 == 0:
        print(
            f"[step {global_step_counter['n']}] "
            f"reward={training_log['mean_reward'][-1]:.1f}  "
            f"grader={training_log['mean_grader'][-1]:.3f}  "
            f"delivery={training_log['delivery_rate'][-1]:.2%}  "
            f"valid={training_log['valid_rate'][-1]:.2%}  "
            f"profit={training_log['mean_profit'][-1]:.1f}  "
            f"carbon_eff={training_log['carbon_efficiency'][-1]:.1f}"
        )

    return rewards

In [14]:
!rm -rf /content/unsloth_compiled_cache

In [ ]:
# ── NEW CELL — Dataset rebuild + GRPOTrainer ─────────────────────────────────

from trl import GRPOConfig, GRPOTrainer

# ── 1. Rebuild the training dataset with DIVERSE episode-level prompts ───────
#
# Tier 1 fix: instead of 10 prompts tiled to 100 (overfitting trap), we sample
# N_UNIQUE_PROMPTS distinct initial states across TRAIN_TASK_IDS and
# TRAIN_SEED_RANGE. Each dataset entry is still one full-episode prompt; the
# model's completion is still a 10-action JSON array; env_reward_func still
# runs the full 10-step episode.

INTER_EPISODE_DELAY = 2.0  # Slow down to give the Space time to breathe

def collect_episode_prompts(n_unique: int = N_UNIQUE_PROMPTS) -> list[dict]:
    """Collect `n_unique` distinct initial-state prompts.

    Strategy (robust whether or not the env honors `seed`):
      - Rotate through TRAIN_TASK_IDS (different tasks = different initial state distributions)
      - Pass seed to /reset (env may or may not use it — doesn't matter)
      - Apply world_wrapper with seed-derived rng so the prompt is perturbed
        client-side in a deterministic, seed-specific way
    """
    prompts = []
    seed_start, seed_end = TRAIN_SEED_RANGE
    seeds = list(range(seed_start, seed_end))[:n_unique]

    for i, seed in enumerate(seeds):
        task_id = TRAIN_TASK_IDS[i % len(TRAIN_TASK_IDS)]
        session_id = f"ep-{uuid.uuid4().hex[:12]}"
        headers    = {"X-Session-Id": session_id}
        try:
            # Pass seed in payload — if env supports it, great; if not, we
            # still get diversity from the client-side wrapper below.
            r = _post_with_retry(
                f"{ENV_URL}/reset",
                json={"task_id": task_id, "seed": seed},
                headers=headers, timeout=30
            )
            obs = r.json()
            wrapped_obs, _ = apply_world_wrapper(obs, random.Random(seed))
            user_prompt  = format_observation_prompt(wrapped_obs)
            full_prompt  = (
                f"<|system|>\n{SYSTEM_PROMPT}\n"
                f"<|user|>\n{user_prompt}\n"
                f"<|assistant|>\n"
            )
            prompts.append({"prompt": full_prompt, "_task_id": task_id, "_seed": seed})
        except Exception as e:
            print(f"[collect_episode_prompts] seed {seed} / {task_id} skipped: {e}")
        _time.sleep(INTER_EPISODE_DELAY)
    return prompts


print(f"Collecting {N_UNIQUE_PROMPTS} DIVERSE episode prompts "
      f"(seeds {TRAIN_SEED_RANGE[0]}-{TRAIN_SEED_RANGE[1]-1} × {len(TRAIN_TASK_IDS)} tasks)...")
episode_prompts = collect_episode_prompts(n_unique=N_UNIQUE_PROMPTS)
print(f"Collected {len(episode_prompts)} unique prompts.")

# Sanity-check diversity
unique_prompt_texts = {p["prompt"] for p in episode_prompts}
print(f"Unique prompt strings: {len(unique_prompt_texts)} "
      f"({'OK' if len(unique_prompt_texts) >= N_UNIQUE_PROMPTS * 0.8 else 'WARN — low diversity'})")

# Repeat (not tile identical) to fill MAX_STEPS if needed — each GRPO step
# consumes one prompt. If we have 50 unique and MAX_STEPS=100, each prompt
# gets seen twice on average, which is fine.
if len(episode_prompts) < MAX_STEPS:
    import random as _rpy_ds
    _rng_ds = _rpy_ds.Random(42)
    extended = list(episode_prompts)
    while len(extended) < MAX_STEPS:
        extended.append(_rng_ds.choice(episode_prompts))
    episode_prompts = extended
episode_prompts = episode_prompts[:MAX_STEPS]
print(f"Final training dataset: {len(episode_prompts)} prompts "
      f"({len(unique_prompt_texts)} unique)")

from datasets import Dataset
# Drop the debug metadata columns — GRPOTrainer only wants "prompt"
train_dataset = Dataset.from_list([{"prompt": p["prompt"]} for p in episode_prompts])

# ── 2. GRPOConfig ─────────────────────────────────────────────────────────────
grpo_config = GRPOConfig(
    output_dir                  = f"/content/drive/MyDrive/grpo-eco-logistics-{TASK_ID}",
    max_prompt_length           = 1024,
    max_completion_length       = 2048,
    per_device_train_batch_size = PER_DEVICE_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    num_generations             = NUM_GENERATIONS,
    learning_rate               = LEARNING_RATE,
    max_steps                   = MAX_STEPS,
    logging_steps               = 5,
    save_steps                  = 50,
    report_to                   = "none",
    seed                        = 42,
    remove_unused_columns       = False,
)

# ── 3. GRPOTrainer + train ────────────────────────────────────────────────────
#
# NOTE: env_reward_func is called by GRPOTrainer during .train(). Do NOT call
# eager_generate or FastLanguageModel.for_inference(...) while .train() is
# running — it toggles the LoRA adapter state and corrupts gradients.

trainer = GRPOTrainer(
    model         = model,
    args          = grpo_config,
    reward_funcs  = [env_reward_func],
    train_dataset = train_dataset,
    tokenizer     = tokenizer,
)

print(
    f"\nStarting GRPO training — {MAX_STEPS} steps, "
    f"effective batch {PER_DEVICE_BATCH * GRAD_ACCUM_STEPS}, "
    f"{NUM_GENERATIONS} generations per prompt.\n"
    f"max_completion_length = {grpo_config.max_completion_length} (≈ 10 JSON action objects)\n"
    f"Dataset: {len(train_dataset)} prompts, {len(unique_prompt_texts)} unique.\n"
)
trainer.train()
print("Training complete.")


Dataset: 100 prompts from 10 unique initial states

Starting GRPO training — 100 steps, effective batch 4, 4 generations per prompt.
max_completion_length = 2048 (≈ 10 JSON action objects)



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss


## 9. Post-training evaluation

In [ ]:
print(f"Running 10 AFTER-TRAINING episodes on HELD-OUT seeds "
      f"{EVAL_SEED_RANGE[0]}-{EVAL_SEED_RANGE[1]-1} (not seen in training)...")
eval_seeds = list(range(*EVAL_SEED_RANGE))
trained_results = [run_episode(eager_generate, seed=s) for s in eval_seeds]

# NOTE: baseline_results, random_results, heuristic_results were already run
# on seeds 500-509 in cell 19 — the SAME held-out seeds. This is a fair
# apples-to-apples comparison on unseen initial states.

summarize(random_results,    "RANDOM POLICY")
summarize(heuristic_results, "HEURISTIC POLICY")
summarize(baseline_results,  "BEFORE TRAINING (base Qwen)")
summarize(trained_results,   "AFTER TRAINING (GRPO Qwen)")

print("\n=== DELTAS (After − Before, on held-out seeds) ===")
for key in ["total_reward", "profit", "carbon_used",
            "carbon_efficiency", "delivery_success_rate", "grader_score"]:
    before = np.mean([r[key] for r in baseline_results])
    after  = np.mean([r[key] for r in trained_results])
    print(f"  {key:25s}: {before:+8.3f} → {after:+8.3f}  (Δ {after-before:+.3f})")

print("\n=== AFTER vs ALL BASELINES (grader score) ===")
print(f"  Random policy     : {np.mean([r['grader_score'] for r in random_results]):.3f}")
print(f"  Heuristic policy  : {np.mean([r['grader_score'] for r in heuristic_results]):.3f}")
print(f"  Base Qwen         : {np.mean([r['grader_score'] for r in baseline_results]):.3f}")
print(f"  GRPO Qwen (ours)  : {np.mean([r['grader_score'] for r in trained_results]):.3f}")


## 10. Training curves for the pitch deck

In [ ]:
# ── Cell 59 (REPLACEMENT — 4-way comparison) ─────────────────────────────────
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

log_df = pd.DataFrame(training_log)
log_df.to_csv(f"training_log_{TASK_ID}.csv", index=False)
print(f"Saved training_log_{TASK_ID}.csv  ({len(log_df)} GRPO steps logged)")

WINDOW = 5

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# ── (1) Reward vs GRPO step ────────────────────────────────────────────────────
axes[0, 0].plot(log_df["grpo_step"], log_df["mean_reward"],
                alpha=0.4, label="per step")
axes[0, 0].plot(log_df["grpo_step"],
                log_df["mean_reward"].rolling(WINDOW, min_periods=1).mean(),
                linewidth=2, label=f"{WINDOW}-step rolling mean")
axes[0, 0].set_title(f"Reward vs GRPO step — {TASK_ID}")
axes[0, 0].set_xlabel("GRPO step")
axes[0, 0].set_ylabel("Mean episode reward")
axes[0, 0].legend()

# ── (2) Carbon efficiency vs GRPO step ────────────────────────────────────────
axes[0, 1].plot(log_df["grpo_step"], log_df["carbon_efficiency"],
                alpha=0.4, color="tab:green", label="per step")
axes[0, 1].plot(log_df["grpo_step"],
                log_df["carbon_efficiency"].rolling(WINDOW, min_periods=1).mean(),
                linewidth=2, color="tab:green", label=f"{WINDOW}-step rolling mean")
axes[0, 1].set_title("Carbon efficiency vs GRPO step")
axes[0, 1].set_xlabel("GRPO step")
axes[0, 1].set_ylabel("Profit per unit carbon")
axes[0, 1].legend()

# ── (3) 4-way grader score: Random / Heuristic / Before / After ───────────────
labels  = ["Random", "Heuristic", "Base Qwen", "GRPO (ours)"]
results = [random_results, heuristic_results, baseline_results, trained_results]
means   = [np.mean([r["grader_score"] for r in rs]) for rs in results]
stds    = [np.std([r["grader_score"] for r in rs])  for rs in results]
colors  = ["#d9d9d9", "#a0a0a0", "#888888", "#2a9d8f"]

axes[1, 0].bar(labels, means, yerr=stds, capsize=6, color=colors)
axes[1, 0].set_ylabel("Grader score (0–1)")
axes[1, 0].set_title("Task completion: 4-way comparison (held-out seeds)")
axes[1, 0].set_ylim(0, 1)
for i, v in enumerate(means):
    axes[1, 0].text(i, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)

# ── (4) 4-way delivery rate ───────────────────────────────────────────────────
delivery_means = [np.mean([r["delivery_success_rate"] for r in rs]) for rs in results]
delivery_stds  = [np.std([r["delivery_success_rate"] for r in rs])  for rs in results]
axes[1, 1].bar(labels, delivery_means, yerr=delivery_stds, capsize=6, color=colors)
axes[1, 1].set_ylabel("Delivery success rate")
axes[1, 1].set_title("Delivery rate: 4-way comparison (held-out seeds)")
axes[1, 1].set_ylim(0, 1)
for i, v in enumerate(delivery_means):
    axes[1, 1].text(i, v + 0.02, f"{v:.1%}", ha="center", fontsize=9)

plt.tight_layout()
fig.savefig(f"training_curves_{TASK_ID}.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved training_curves_{TASK_ID}.png")


## 11. Save qualitative trajectories (before / after example runs)

Two text files the judges can read — one baseline episode and one trained episode — showing the agent's actual decision-making.

In [ ]:
def dump_trajectory(result, path):
    with open(path, "w") as f:
        f.write(f"Task: {TASK_ID}\n")
        f.write(f"Total reward: {result['total_reward']:.2f}\n")
        f.write(f"Profit: {result['profit']:.2f}\n")
        f.write(f"Carbon used: {result['carbon_used']:.2f}\n")
        f.write(f"Delivery rate: {result['delivery_success_rate']:.1%}\n")
        f.write(f"Grader score: {result['grader_score']:.3f}\n\n")
        for t in result["trajectory"]:
            flags = []
            if t["shocked"]: flags.append("DEMAND_SHOCK")
            if t["contested"]: flags.append("COMPETITOR_BID")
            flag_str = f" [{', '.join(flags)}]" if flags else ""
            f.write(f"Step {t['step']}{flag_str}: {t['action']}\n")
            f.write(f"  reward={t['reward']:+.2f}  profit={t['profit']:+.2f}  carbon={t['carbon']:.2f}  valid={t['valid']}\n")
    print(f"Wrote {path}")

# Use the median-performing episode from each set (more representative than best/worst)
def median_idx(results, key="total_reward"):
    vals = [r[key] for r in results]
    sorted_idx = sorted(range(len(vals)), key=lambda i: vals[i])
    return sorted_idx[len(sorted_idx) // 2]

dump_trajectory(baseline_results[median_idx(baseline_results)], f"trajectory_before_{TASK_ID}.txt")
dump_trajectory(trained_results[median_idx(trained_results)], f"trajectory_after_{TASK_ID}.txt")

## 12. Save + push the LoRA adapter

In [ ]:
model.save_pretrained(f"eco-logistics-lora-v2-{TASK_ID}")
tokenizer.save_pretrained(f"eco-logistics-lora-v2-{TASK_ID}")

# Optional: push to HF Hub
# model.push_to_hub(f"YOUR_USERNAME/eco-logistics-lora-v2-{TASK_ID}")
# tokenizer.push_to_hub(f"YOUR_USERNAME/eco-logistics-lora-v2-{TASK_ID}")

## What's different from v2 (Tier 1 patches)

| Change | Why |
|---|---|
| **Dataset diversity: 50 unique prompts across 3 tasks** (was 10 tiled to 100) | Fixes the overfit trap — training saw varied initial states, not a memorized loop |
| **Explicit train/eval seed split** (train 0-49, eval 500-509) | Generalization claim is now testable: AFTER numbers are on seeds training never saw |
| **Random + heuristic policy baselines** added alongside base Qwen | Kills the "is the delta just JSON parsing?" question — 4-way comparison |
| **4-way grader + delivery bar charts** | Visual proof that GRPO Qwen > base Qwen > heuristic > random |
| **Bug fixes**: `SAFE_FALLBACK_ACTIONS` aliasing, `max_completion_length` lie, `torchcodec` order, `eager_generate` safety note, concurrency count (16 not 8) | Removes landmines that would have surfaced at 2am on hackathon day |

## Runbook for on-site

1. Set `HF_TOKEN`, confirm `ENV_URL` is live, confirm Space has `max_concurrent_envs >= 16`
2. Run cells 1–8 (setup + smoke test). Smoke test must pass before continuing.
3. Run cells 9–19 (wrappers + 3 baselines). You'll get 30 baseline-eval episodes.
4. Run cells 20–23 (GRPO train). ~45 min on T4, ~20 min on A100.
5. Run cells 24–29 (AFTER eval + plots + trajectories).
6. Re-run on `net_zero_profit` — this is where the storytelling win lives.
7. Build pitch from `training_curves_*.png` + the delta printout.

## Files produced for the pitch

- `training_curves_{TASK_ID}.png` — the 4-panel money shot
- `training_log_{TASK_ID}.csv` — raw metrics (in case judges want to inspect)
- `trajectory_before_{TASK_ID}.txt`, `trajectory_after_{TASK_ID}.txt` — qualitative evidence
- LoRA adapter saved to `/content/drive/MyDrive/grpo-eco-logistics-{TASK_ID}`
